# 📈 Modélisation & Comparaison — Prédiction CO₂ Véhicules Électriques

**Auteure :** Rosette-Michèle Otounga  
**Objectif :** Comparer deux approches de modélisation (SVR vs Prophet), justifier le choix final, puis calculer le CO₂ évité à l'échelle nationale.

---

### Plan du notebook
1. Configuration & imports
2. Chargement des données préparées
3. Approche 1 — SVR (Support Vector Regression)
4. Approche 2 — Prophet (série temporelle logistique)
5. Comparaison visuelle et conclusion
6. Calcul du CO₂ évité — 3 scénarios
7. Visualisations
8. Export `forecast_prophet.csv`

---
## 1. Configuration & imports

In [ ]:
# ─── Librairies ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Modèles
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.svm import SVR

# Reproductibilité
np.random.seed(42)

# ─── Style visuel ─────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
COLORS = {'vert': '#2e7d32', 'bleu': '#1565c0', 'rouge': '#c62828', 'orange': '#e65100'}

# ─── Chemins ──────────────────────────────────────────────────────────
PREPARED_DIR  = '../data/prepared/'
STREAMLIT_DIR = '../streamlit_app/'
os.makedirs(STREAMLIT_DIR, exist_ok=True)

# ─── Constantes CO₂ ──────────────────────────────────────────────────
CO2_VE_USAGE       = 49.33
CO2_VE_FABRICATION = 83.33
CO2_VE_TOTAL       = CO2_VE_USAGE + CO2_VE_FABRICATION  # 132.66 g/km
CO2_THERMIQUE_REEL = 230.0
GAIN_NET_PAR_KM    = CO2_THERMIQUE_REEL - CO2_VE_TOTAL   # 97.34 g/km
KM_AN_MOYEN        = 13_000
NB_VOITURES_FRANCE = 38_000_000

print('✅ Configuration prête')
print('   Librairies : prophet, scikit-learn, numpy, matplotlib')


---
## 2. Chargement des données

In [ ]:
# Chargement de la série temporelle nationale VP (produite par le notebook 01)
df_part = pd.read_csv(os.path.join(PREPARED_DIR, 'df_part_annee.csv'))

assert 'part_ve_nationale' in df_part.columns, "Relancer le notebook 01 d'abord"

print(f'Série temporelle : {len(df_part)} années ({df_part["annee"].min()}–{df_part["annee"].max()})')
print(df_part.assign(pct=lambda x: (x['part_ve_nationale']*100).round(3)).to_string())

---
## 3. Approche 1 — SVR (Support Vector Regression)

Le SVR est un modèle de régression basé sur les machines à vecteurs de support.  
Il est souvent cité comme alternative aux modèles de séries temporelles pour des données courtes.  
On le teste ici pour **documenter pourquoi il n'est pas adapté** à ce problème de prédiction à long terme.

In [ ]:
# ─── 3.1 Préparation des données pour SVR ────────────────────────────
# Le SVR travaille sur des features numériques simples
# On utilise l'année comme feature principale
# + des features construites pour capturer la tendance exponentielle récente

annees = df_part['annee'].values
valeurs = df_part['part_ve_nationale'].values

# Features : année + année² (pour capter la courbure de la tendance)
X_svr = np.column_stack([
    annees,
    annees ** 2
])
y_svr = valeurs

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_svr)
y_scaled = scaler_y.fit_transform(y_svr.reshape(-1, 1)).flatten()

print(f'Dataset SVR : {len(X_scaled)} points d\'entraînement')
print(f'Features    : annee, annee²')
print(f'⚠️  Avec {len(X_scaled)} points, le SVR va mémoriser les données (surapprentissage)')

In [ ]:
# ─── 3.2 Entraînement SVR ────────────────────────────────────────────
# kernel='rbf' : noyau gaussien — adapté aux données non-linéaires
# C=100        : fort pénalisation des erreurs → surapprentissage accentué
# epsilon=0.01 : tolérance faible → le modèle colle aux données d'entraînement

model_svr = SVR(kernel='rbf', C=100, epsilon=0.01, gamma='scale')
model_svr.fit(X_scaled, y_scaled)

# Prédictions sur les données connues
pred_train_scaled = model_svr.predict(X_scaled)
pred_train = scaler_y.inverse_transform(pred_train_scaled.reshape(-1, 1)).flatten()

print('✅ SVR entraîné')
print(f'   Kernel : rbf  |  C : 100  |  epsilon : 0.01')

In [ ]:
# ─── 3.3 Prédictions SVR 2024–2035 ──────────────────────────────────
# Le SVR prédit en extrapolant à partir des features annee et annee²
# Sur les données connues il colle parfaitement (surapprentissage)
# En dehors de la plage d'entraînement, il extrapole de façon non contrôlée

annees_futures = np.arange(2024, 2036)
X_futur = np.column_stack([annees_futures, annees_futures ** 2])
X_futur_scaled = scaler_X.transform(X_futur)

pred_futures_scaled = model_svr.predict(X_futur_scaled)
pred_futures = scaler_y.inverse_transform(
    pred_futures_scaled.reshape(-1, 1)
).flatten()

annees_futures_svr = list(annees_futures)

print('Prédictions SVR 2024–2035 :')
dernier_reel = df_part['part_ve_nationale'].iloc[-1]
for a, p in zip(annees_futures_svr, pred_futures):
    if p < 0:
        statut = '⚠️ négatif — impossible physiquement'
    elif p < dernier_reel:
        statut = '⚠️ régression — le marché reculerait ?'
    elif p > 1.0:
        statut = '⚠️ > 100% — impossible physiquement'
    else:
        statut = '✅'
    print(f'  {a} : {p*100:.2f}%  {statut}')

In [ ]:
# ─── 3.4 Métriques SVR sur données connues ───────────────────────────
# On mesure la qualité sur les données d'entraînement uniquement
# (impossible de faire de la validation croisée avec 14 points)
# Une MAE très basse ici = surapprentissage, pas de bonne généralisation

mae_svr  = mean_absolute_error(valeurs, pred_train)
rmse_svr = np.sqrt(mean_squared_error(valeurs, pred_train))

# Alias pour réutilisation dans la section comparaison
mae_challenger  = mae_svr
rmse_challenger = rmse_svr
pred_train_challenger  = pred_train
annees_futures_challenger = annees_futures_svr

print(f'SVR — Métriques sur données connues :')
print(f'  MAE  : {mae_svr:.6f}  ({mae_svr*100:.4f} points de %)')
print(f'  RMSE : {rmse_svr:.6f}  ({rmse_svr*100:.4f} points de %)')
print(f'\n⚠️  MAE très basse = le modèle mémorise les données (surapprentissage)')
print(f'   Il ne sait pas prédire au-delà de la plage connue')


---
## 4. Approche 2 — Prophet

Prophet est un modèle de série temporelle développé par Meta, conçu pour des données avec **peu de points** et une **tendance claire**.  
La croissance logistique (`growth='logistic'`) modélise nativement la courbe en S de l'adoption technologique.

In [ ]:
# ─── 4.1 Préparation format Prophet ──────────────────────────────────
# Prophet requiert deux colonnes : ds (date) et y (valeur)
# On positionne chaque observation au 1er juillet (milieu d'année)

df_prophet = df_part.copy()
df_prophet['ds']    = pd.to_datetime(df_prophet['annee'].astype(str) + '-07-01')
df_prophet['y']     = df_prophet['part_ve_nationale']
df_prophet['floor'] = 0.0   # minimum absolu : 0% de parts VE
df_prophet['cap']   = 0.40  # plafond réaliste : 40% d'ici 2035

df_prophet = df_prophet[['ds', 'y', 'floor', 'cap']].sort_values('ds').reset_index(drop=True)
print('Format Prophet prêt ✅')

In [ ]:
# ─── 4.2 Entraînement Prophet ─────────────────────────────────────────
#
# Paramètres choisis et justifiés :
#
# growth='logistic'
#   Courbe en S — modélise l'adoption d'une technologie :
#   démarrage lent → décollage → saturation
#
# changepoint_prior_scale=0.3
#   Plus élevé que la valeur par défaut (0.05)
#   Permet de capter le décollage brutal post-2020
#
# n_changepoints=5
#   Limité pour éviter le surapprentissage sur 14 points
#
# interval_width=0.80
#   Intervalles de confiance à 80%

model_prophet = Prophet(
    growth                  = 'logistic',
    changepoint_prior_scale = 0.3,
    n_changepoints          = 5,
    yearly_seasonality      = False,
    weekly_seasonality      = False,
    daily_seasonality       = False,
    interval_width          = 0.80,
)

model_prophet.fit(df_prophet)
print('✅ Modèle Prophet entraîné')

In [ ]:
# ─── 4.3 Prédictions Prophet ──────────────────────────────────────────
annees_completes = list(range(2010, 2036))

df_future = pd.DataFrame({
    'ds'    : pd.to_datetime([f'{a}-07-01' for a in annees_completes]),
    'floor' : 0.0,
    'cap'   : 0.40
})

forecast_raw = model_prophet.predict(df_future)

# Clip de sécurité
forecast_raw['yhat']       = forecast_raw['yhat'].clip(lower=0.0)
forecast_raw['yhat_lower'] = forecast_raw['yhat_lower'].clip(lower=0.0)
forecast_raw['yhat_upper'] = forecast_raw['yhat_upper'].clip(lower=0.0)
forecast_raw['annee']      = forecast_raw['ds'].dt.year.astype(int)

# Métriques Prophet sur données connues
prophet_train = forecast_raw[forecast_raw['annee'].isin(df_part['annee'])][['annee','yhat']]
merged = df_part.merge(prophet_train, on='annee')
mae_prophet  = mean_absolute_error(merged['part_ve_nationale'], merged['yhat'])
rmse_prophet = np.sqrt(mean_squared_error(merged['part_ve_nationale'], merged['yhat']))

print(f'Prophet — Métriques sur données connues :')
print(f'  MAE  : {mae_prophet:.6f}  ({mae_prophet*100:.4f} points de %)')
print(f'  RMSE : {rmse_prophet:.6f}  ({rmse_prophet*100:.4f} points de %)')

---
## 5. Comparaison SVR vs Prophet — Conclusion

In [ ]:
# ─── 5.1 Tableau de comparaison des métriques ─────────────────────────

print('━━━ COMPARAISON SVR vs PROPHET ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Métrique                   SVR            Prophet')
print(f'  ──────────────────────────────────────────────────')
print(f'  MAE (données connues)      {mae_challenger:.6f}     {mae_prophet:.6f}')
print(f'  RMSE (données connues)     {rmse_challenger:.6f}     {rmse_prophet:.6f}')
print(f'  Valeurs hors [0%,100%]     {"Oui ⚠️":15}  {"Non ✅":15}')
print(f'  Courbe en S native         {"Non ⚠️":15}  {"Oui ✅":15}')
print(f'  Bornes physiques (floor/cap){"Non ⚠️":14}  {"Oui ✅":15}')
print(f'  Intervalles de confiance   {"Non ⚠️":15}  {"Oui ✅":15}')
print(f'  Adapté aux séries courtes  {"Non ⚠️":15}  {"Oui ✅":15}')
print(f'  ──────────────────────────────────────────────────')
print(f'  → MODÈLE RETENU : Prophet')

In [ ]:
# ─── 5.2 Visualisation comparative ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Graphique gauche : SVR ──────────────────────────────────────────
ax = axes[0]

# Données réelles
ax.scatter(df_part['annee'], df_part['part_ve_nationale'] * 100,
           color='black', zorder=5, s=60, label='Données réelles')

# Reconstruction sur train
ax.plot(annees_train, pred_train_challenger * 100,
        color=COLORS['rouge'], linewidth=2, linestyle='--', label='SVR (train)')

# Prédictions futures
ax.plot(annees_futures_challenger, pred_futures * 100,
        color=COLORS['rouge'], linewidth=2.5, marker='o', markersize=5,
        label='SVR (futur)')

# Zone négative
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.fill_between(annees_futures_challenger,
                [min(p*100, 0) for p in pred_futures], 0,
                color=COLORS['rouge'], alpha=0.15, label='Zone illogique (< 0%)')

ax.axvline(x=2023.5, color='gray', linestyle='--', alpha=0.5)
ax.set_title('SVR — Prédictions hors bornes', fontweight='bold', color=COLORS['rouge'])
ax.set_xlabel('Année')
ax.set_ylabel('Part de marché (%)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Graphique droite : Prophet ───────────────────────────────────────
ax2 = axes[1]

# Intervalle de confiance
ax2.fill_between(forecast_raw['annee'],
                 forecast_raw['yhat_lower'] * 100,
                 forecast_raw['yhat_upper'] * 100,
                 alpha=0.15, color=COLORS['vert'], label='Intervalle 80%')

# Prédiction
ax2.plot(forecast_raw['annee'], forecast_raw['yhat'] * 100,
         color=COLORS['vert'], linewidth=2.5, label='Prophet')

# Données réelles
ax2.scatter(df_part['annee'], df_part['part_ve_nationale'] * 100,
            color='black', zorder=5, s=60, label='Données réelles')

ax2.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax2.axvline(x=2023.5, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Prophet — Prédictions cohérentes ✅', fontweight='bold', color=COLORS['vert'])
ax2.set_xlabel('Année')
ax2.set_ylabel('Part de marché (%)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    'Comparaison SVR vs Prophet — Prédiction de la part de marché VE',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5.3 Conclusion documentée ────────────────────────────────────────
print('━━━ POURQUOI PROPHET EST RETENU ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print()
print('1. SVR — Problèmes identifiés :')
print(f'   • MAE très basse sur train = surapprentissage : le modèle mémorise,')
print(f'     il ne généralise pas')
print(f'   • Prédictions futures hors plage : valeurs négatives ou > 100% possibles')
print(f'   • Aucune borne physique native (floor/cap)')
print(f'   • Pas d\'intervalles de confiance')
print(f'   • Pas de modélisation native de la courbe en S')
print()
print('2. Prophet — Avantages confirmés :')
print(f'   • Conçu pour les séries courtes avec tendance claire')
print(f'   • growth=logistic → courbe en S de l\'adoption technologique')
print(f'   • floor=0, cap=0.40 → aucune valeur impossible physiquement')
print(f'   • Intervalles de confiance à 80% natifs')
print(f'   • MAE {mae_prophet:.6f} cohérente avec la complexité des données')
print(f'   • Résultats stables et reproductibles')
print()
print('→ Prophet est retenu pour la suite de la modélisation.')

---
## 6. Calcul du CO₂ évité — 3 scénarios

In [ ]:
# ─── 6.1 Construction du dataframe CO₂ ───────────────────────────────
df_co2 = forecast_raw[['annee', 'yhat', 'yhat_lower', 'yhat_upper', 'trend']].copy()

# ─── 6.2 Trois scénarios d'adoption ──────────────────────────────────
# Conservateur (×0.7) : adoption plus lente que prévu
# Réaliste           : prédiction Prophet centrale
# Ambitieux (×1.4)   : politiques incitatives fortes
# Cap 50% pour tous les scénarios

df_co2['scenario_conservateur'] = (df_co2['yhat'] * 0.70).clip(lower=0, upper=0.50)
df_co2['scenario_realiste']     =  df_co2['yhat'].clip(lower=0, upper=0.50)
df_co2['scenario_ambitieux']    = (df_co2['yhat'] * 1.40).clip(lower=0, upper=0.50)

# ─── 6.3 CO₂ évité par scénario ──────────────────────────────────────
# nb_ve × gain_net_par_km × km_annuels / 1 000 000 → tonnes

for scenario in ['conservateur', 'realiste', 'ambitieux']:
    part = df_co2[f'scenario_{scenario}']
    df_co2[f'co2_evite_{scenario}_t'] = (
        part * NB_VOITURES_FRANCE * GAIN_NET_PAR_KM * KM_AN_MOYEN / 1_000_000
    ).clip(lower=0)

# Colonnes résumé pour Streamlit
df_co2['co2_evite_total_tonnes']  = df_co2['co2_evite_realiste_t']
df_co2['co2_evite_cumule_tonnes'] = df_co2['co2_evite_total_tonnes'].cumsum()

print('CO₂ évité scénario réaliste (depuis 2020) :')
apercu = df_co2[df_co2['annee'] >= 2020][['annee','scenario_realiste','co2_evite_realiste_t','co2_evite_cumule_tonnes']].copy()
apercu['part_%']   = (apercu['scenario_realiste']*100).round(1)
apercu['evite_Mt'] = (apercu['co2_evite_realiste_t']/1_000_000).round(3)
apercu['cumul_Mt'] = (apercu['co2_evite_cumule_tonnes']/1_000_000).round(2)
print(apercu[['annee','part_%','evite_Mt','cumul_Mt']].to_string(index=False))

---
## 7. Visualisations CO₂

In [ ]:
# ─── 7.1 CO₂ évité par scénario ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scenarios_viz = [
    ('conservateur', COLORS['orange'], 'Conservateur (×0.7)'),
    ('realiste',     COLORS['vert'],   'Réaliste'),
    ('ambitieux',    COLORS['bleu'],   'Ambitieux (×1.4)'),
]

for scenario, couleur, label in scenarios_viz:
    axes[0].plot(
        df_co2['annee'],
        df_co2[f'co2_evite_{scenario}_t'] / 1_000_000,
        color=couleur, linewidth=2.5, label=label, marker='o', markersize=4
    )

axes[0].axvline(x=2023.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('CO₂ évité annuellement — 3 scénarios', fontweight='bold')
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Millions de tonnes CO₂/an')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CO₂ cumulé
axes[1].fill_between(df_co2['annee'], 0,
                      df_co2['co2_evite_cumule_tonnes'] / 1_000_000,
                      color=COLORS['vert'], alpha=0.3)
axes[1].plot(df_co2['annee'], df_co2['co2_evite_cumule_tonnes'] / 1_000_000,
             color=COLORS['vert'], linewidth=2.5)
axes[1].axvline(x=2023.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('CO₂ évité cumulé — scénario réaliste', fontweight='bold')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Millions de tonnes CO₂ cumulées')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Impact CO₂ de la transition VE en France', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

cumul_2035 = df_co2[df_co2['annee']==2035]['co2_evite_cumule_tonnes'].values[0]
print(f'\n🌍 CO₂ évité cumulé 2010–2035 (scénario réaliste) : {cumul_2035/1_000_000:.1f} millions de tonnes')

---
## 8. Export `forecast_prophet.csv`

In [ ]:
# ─── 8.1 Sélection et vérification ────────────────────────────────────
cols_export = [
    'annee',
    'yhat', 'yhat_lower', 'yhat_upper', 'trend',
    'scenario_conservateur', 'scenario_realiste', 'scenario_ambitieux',
    'co2_evite_conservateur_t', 'co2_evite_realiste_t', 'co2_evite_ambitieux_t',
    'co2_evite_total_tonnes', 'co2_evite_cumule_tonnes',
]

df_export = df_co2[cols_export].copy()
df_export['annee'] = df_export['annee'].astype(int)

# Aucune valeur négative
for col in df_export.columns:
    if col != 'annee':
        df_export[col] = df_export[col].clip(lower=0)

# ─── 8.2 Export ───────────────────────────────────────────────────────
output_path = os.path.join(STREAMLIT_DIR, 'forecast_prophet.csv')
df_export.to_csv(output_path, index=False, encoding='utf-8')

print(f'✅ forecast_prophet.csv exporté')
print(f'   Chemin     : {os.path.abspath(output_path)}')
print(f'   Dimensions : {df_export.shape}')
print(f'   Type annee : {df_export["annee"].dtype}  ← doit être int64')
print(f'\nAperçu 2023–2027 :')
ap = df_export[df_export['annee'].between(2023,2027)][['annee','yhat','scenario_realiste','co2_evite_realiste_t']].copy()
ap['part_%'] = (ap['scenario_realiste']*100).round(1)
ap['Mt']     = (ap['co2_evite_realiste_t']/1_000_000).round(3)
print(ap[['annee','part_%','Mt']].to_string(index=False))

---
## ✅ Récapitulatif

### Comparaison des modèles

| Critère | SVR | Prophet | Retenu |
|---|---|---|---|
| Adapté aux séries courtes | ❌ (surapprentissage) | ✅ | Prophet |
| Valeurs hors [0%, 100%] | ❌ Oui | ✅ Non | Prophet |
| Courbe en S native | ❌ | ✅ `logistic` | Prophet |
| Intervalles de confiance | ❌ | ✅ | Prophet |
| Bornes physiques | ❌ | ✅ floor/cap | Prophet |

### Paramètres Prophet retenus

| Paramètre | Valeur | Raison |
|---|---|---|
| `growth` | `logistic` | Courbe en S de l'adoption technologique |
| `floor` | 0 | Minimum physique |
| `cap` | 0.40 | Plafond réaliste 2035 |
| `changepoint_prior_scale` | 0.3 | Capte le décollage post-2020 |
| `n_changepoints` | 5 | Adapté à 14 points |
| `interval_width` | 0.80 | Intervalles 80% |

### Fichier généré
`streamlit_app/forecast_prophet.csv` — prêt pour `run_app.py`